# Notebook 1: Data Acquisition
**Purpose:** Fetch heavy metal oxide data from the Materials Project database via `mp-api`, apply initial thermodynamic/structural filters, and save to a CSV file for downstream tasks.

In [3]:
import os
import pandas as pd
from dotenv import load_dotenv
from mp_api.client import MPRester
from pymatgen.core import Composition

# Load API key
load_dotenv()
API_KEY = os.getenv("MP_API_KEY")

if not API_KEY:
    raise ValueError("MP_API_KEY not found in .env file!")

# Target metals
metals = ["Sn", "Hf", "Zr", "Ti", "Al", "Zn", "Ni"]

# Fetch data
mpr = MPRester(API_KEY)
all_docs = []

for metal in metals:
    docs = mpr.materials.summary.search(
        elements=[metal, "O"],
        band_gap=(0.1, None),
        fields=[
            "material_id", "formula_pretty", "band_gap",
            "formation_energy_per_atom", "density", "volume",
            "symmetry", "elements", "nsites",
            "energy_above_hull", "composition"
        ]
    )
    print(f"{metal}: {len(docs)} compounds")
    all_docs.extend(docs)

# Convert to DataFrame
data = []
for doc in all_docs:
    crystal_system = doc.symmetry.crystal_system if doc.symmetry else 'unknown'
    data.append({
        'material_id': doc.material_id,
        'formula': doc.formula_pretty,
        'composition': doc.composition,
        'band_gap': doc.band_gap,
        'formation_energy_per_atom': doc.formation_energy_per_atom,
        'density': doc.density,
        'volume': doc.volume,
        'nsites': doc.nsites,
        'crystal_system': crystal_system,
        'elements': doc.elements,
        'energy_above_hull': doc.energy_above_hull,
    })

df = pd.DataFrame(data)

# Quality filtering
df = df[df['energy_above_hull'] < 0.1]
df = df[(df['nsites'] >= 4) & (df['nsites'] <= 100)]
df = df[(df['density'] > 1.0) & (df['density'] < 20.0)]

# Add metal_group for GroupKFold
def assign_group(formula, metals):
    try:
        comp = Composition(formula)
    except:
        return "Other"
    for metal in metals:
        if metal in comp:
            return metal
    return "Other"

df['metal_group'] = df['formula'].apply(lambda x: assign_group(x, metals))

# Save
os.makedirs('data', exist_ok=True)
df.to_csv('data/heavy_metal_oxides.csv', index=False)
print(f"\n✅ Saved {len(df)} compounds to data/heavy_metal_oxides.csv")

Retrieving SummaryDoc documents:   0%|          | 0/1826 [00:00<?, ?it/s]

Sn: 1826 compounds


Retrieving SummaryDoc documents:   0%|          | 0/853 [00:00<?, ?it/s]

Hf: 853 compounds


Retrieving SummaryDoc documents:   0%|          | 0/1309 [00:00<?, ?it/s]

...

Al: 2532 compounds


Retrieving SummaryDoc documents:   0%|          | 0/2634 [00:00<?, ?it/s]

Zn: 2634 compounds


Retrieving SummaryDoc documents:   0%|          | 0/2014 [00:00<?, ?it/s]

Ni: 2014 compounds

✅ Saved 7786 compounds to data/heavy_metal_oxides.csv
